# Config

In [12]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [13]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [ ]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Desafío País"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Desafío País"].notna()]

#Eliminar duplicados
df = df.drop_duplicates("Código VRID")

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Desafío País"])
savepath = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)


Test size: 247
Fold 0 - Val size: 330
Archivo guardado exitosamente en /tmp/final_project/dataSplits/desafios/train_test_ids_3folds.json


# 1) TF-IDF

In [14]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [15]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [17]:
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)

split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
savepath = os.path.join(path, "output/desafios/TF_IDF")
model_paths = ["1", "2", "3", "4"]
for idx, des in enumerate(comb_desafios):
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    #Creacion de vectores TFID
    X_train, X_test = gen_TFID_vectors(X_train, X_test)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    for name, model in models_dicc.items():
        print(name)
        results, preds=eval_model(model, X_test, y_test, lang_es)
        print(results)

    savepath_models = os.path.join(savepath, model_paths[idx])
    save_models_and_metrics(savepath_models, results_val, models_dicc, X_test, y_test, df_test, 
                        save_preds=True, lang_es=df_test["Español"], mode_classification="binary")
    

(987, 17354) (247, 17354)
📊 train: Counter({0: 887, 1: 100})
📊 test: Counter({0: 222, 1: 25})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.03}
XGBClassifier: {'mean_test_score': 0.69, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.67, 'std_test_score': 0.05}
LogisticRegression
{'accuracy': 0.8744939271255061, 'f1_macro': 0.6828743010975358, 'cm': array([[204,  18],
       [ 13,  12]]), 'precision': 0.4, 'recall': 0.48, 'f1_es': 0.8765227021040976, 'f1_en': 0.8618986091054192, 'cm_es': array([[99, 15],
       [ 3, 12]]), 'cm_en': array([[105,   3],
       [ 10,   0]])}
RandomForestClassifier
{'accuracy': 0.8785425101214575, 'f1_macro': 0.6096713021491782, 'cm': array([[211,  11],
       [ 19,   6]]), 'precision': 0.35294117647058826, 'recall': 0.24, 'f1_es': 0.84912

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 707, 1: 280})
📊 test: Counter({0: 176, 1: 71})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.72, 'std_test_score': 0.03}
RandomForestClassifier: {'mean_test_score': 0.7, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.68, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.72, 'std_test_score': 0.02}
LogisticRegression
{'accuracy': 0.8016194331983806, 'f1_macro': 0.7376043360433604, 'cm': array([[160,  16],
       [ 33,  38]]), 'precision': 0.7037037037037037, 'recall': 0.5352112676056338, 'f1_es': 0.8673888817844682, 'f1_en': 0.7098229122157338, 'cm_es': array([[92,  8],
       [ 9, 20]]), 'cm_en': array([[68,  8],
       [24, 18]])}
RandomForestClassifier
{'accuracy': 0.7975708502024291, 'f1_macro': 0.7113406881077038, 'cm': array([[166,  10],
 

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 698, 1: 289})
📊 test: Counter({0: 174, 1: 73})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.05}
SVC: {'mean_test_score': 0.69, 'std_test_score': 0.02}
LogisticRegression
{'accuracy': 0.7165991902834008, 'f1_macro': 0.6511299435028248, 'cm': array([[142,  32],
       [ 38,  35]]), 'precision': 0.5223880597014925, 'recall': 0.4794520547945205, 'f1_es': 0.7553523212263741, 'f1_en': 0.660278633795583, 'cm_es': array([[73, 18],
       [14, 24]]), 'cm_en': array([[69, 14],
       [24, 11]])}
RandomForestClassifier
{'accuracy': 0.728744939271255, 'f1_macro': 0.6412140921409214, 'cm': array([[151,  23],
  

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(987, 17354) (247, 17354)
📊 train: Counter({0: 738, 1: 249})
📊 test: Counter({0: 185, 1: 62})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.64, 'std_test_score': 0.03}
LogisticRegression
{'accuracy': 0.631578947368421, 'f1_macro': 0.5444928564190901, 'cm': array([[132,  53],
       [ 38,  24]]), 'precision': 0.3116883116883117, 'recall': 0.3870967741935484, 'f1_es': 0.7692806922660899, 'f1_en': 0.4807102502017756, 'cm_es': array([[94,  8],
       [19,  8]]), 'cm_en': array([[38, 45],
       [19, 16]])}
RandomForestClassifier
{'accuracy': 0.6072874493927125, 'f1_macro': 0.5101310597231594, 'cm': array([[130,  55],
 

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


# 2) SPECTER

In [18]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [19]:
from preprocess.preprocess import one_hot_codification

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

# Crear labels 
df = one_hot_codification(df, "Desafío País", comb_desafios)


In [20]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
#Guardado de resultados
savepath = os.path.join(path, "output/desafios/SPECTER")
model_paths = ["1", "2", "3", "4"]
for des in comb_desafios:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    # Calcular embeddings
    # Parámetros para cargar modelo
    BASE_MODEL = "allenai/specter2_base"
    ADAPTER_NAME="allenai/specter2_classification"
    X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
    X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    for name, model in models_dicc.items():
        print(name)
        results, preds=eval_model(model, X_test, y_test, lang_es)
        print(results)

    savepath_models = os.path.join(savepath, model_paths[idx])
    save_models_and_metrics(savepath_models, results_val, models_dicc, X_test, y_test, df_test, 
                        save_preds=True, lang_es=df_test["Español"], mode_classification="binary")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(987, 768) (247, 768)
📊 train: Counter({0: 738, 1: 249})
📊 test: Counter({0: 185, 1: 62})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.58, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.61, 'std_test_score': 0.01}
LogisticRegression
{'accuracy': 0.6356275303643725, 'f1_macro': 0.5986205402282248, 'cm': array([[116,  69],
       [ 21,  41]]), 'precision': 0.37272727272727274, 'recall': 0.6612903225806451, 'f1_es': 0.7946436727013646, 'f1_en': 0.4712814485039312, 'cm_es': array([[87, 15],
       [12, 15]]), 'cm_en': array([[29, 54],
       [ 9, 26]])}
RandomForestClassifier
{'accuracy': 0.7044534412955465, 'f1_macro': 0.558550618190721, 'cm': array([[158,  27],
       [ 46,  16]]), 'preci

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
